# Hypothesentests für die EDA

**Notebook 4 von 5.** `03_how_a_test_works.ipynb` hat einen einzelnen Test so lange zerlegt, bis
jeder Schritt verständlich war: Hypothesen, eine Teststatistik, ein p-Wert, eine Entscheidung.
Dieses Notebook setzt dieselben fünf Schritte dort ein, wo du sie am meisten brauchen wirst.

Die explorative Datenanalyse (EDA) ist voller Momente, in denen ein Plot ein Muster nahelegt: Eine
Gruppe wirkt größer, zwei Variablen scheinen zusammenzuhängen, eine Kategorie wirkt
überrepräsentiert. Mit einem Hypothesentest prüfst du, ob ein solches Muster mehr als Rauschen
ist, bevor du darüber berichtest oder darauf aufbaust.

Ein Korrelationskoeffizient, Cramérs V oder das Korrelationsverhältnis messen, wie *stark* ein
Muster ist. Ein Test ergänzt die Frage, die diese Zahlen nicht beantworten können: ob das Muster
Zufall sein könnte. Dieses Notebook geht eine EDA-Sitzung an einem echten Datensatz durch. Für
jede Art von Variablenpaar erstellen wir den Plot, quantifizieren das Muster und führen den
passenden Test durch. Unterwegs siehst du beide Ausgänge: Muster, die den Test bestehen, und
Muster, die es nicht tun.

## Lernziele

Am Ende dieses Notebooks solltest du Folgendes können:

- für jede Art von Variablenpaar, die dir in der EDA begegnet, den passenden Hypothesentest
  auswählen.
- Zusammenhänge zwischen numerisch und numerisch, kategorial und numerisch sowie kategorial und
  kategorial mit scipy testen.
- einen Plot, eine Effektstärke und einen p-Wert zu einer Schlussfolgerung über ein Muster
  zusammenführen.
- die Falle des multiplen Testens erkennen, die die explorative Analyse mit sich bringt, und
  explorative Ergebnisse ehrlich berichten.

## Welcher Test für welches Paar?

In der EDA dreht sich jede Frage um ein Paar von Variablen, und die Art des Paars bestimmt den
Test, genauso wie sie das Maß für die Stärke bestimmt:

| Paar von Spalten | Test | scipy-Funktion | Dazu zu berichtende Effektstärke |
| :--- | :--- | :--- | :--- |
| numerisch vs. numerisch | Pearson- oder Spearman-Korrelation | `pearsonr`, `spearmanr` | der Korrelationskoeffizient |
| kategorial mit **zwei** Gruppen vs. numerisch | Welch-t-Test oder Mann-Whitney-U | `ttest_ind`, `mannwhitneyu` | die Differenz der Mittelwerte |
| kategorial mit **drei oder mehr** Gruppen vs. numerisch | einfaktorielle ANOVA | `f_oneway` | das Korrelationsverhältnis |
| kategorial vs. kategorial | Chi-Quadrat-Unabhängigkeitstest | `chi2_contingency` | Cramérs V |

Jeder davon folgt den fünf Schritten aus Notebook 03. Es ändern sich nur die Teststatistik und die
Kurve, mit der sie verglichen wird, und diese Kurven sind die drei aus Abschnitt 5 von Notebook
01: Korrelation und t-Tests vergleichen mit einer t-Verteilung, die ANOVA mit F und der
Chi-Quadrat-Test mit Chi-Quadrat. Mann-Whitney-U arbeitet stattdessen mit Rängen, und um seine
Kurve kümmert sich scipy für dich. Jeder Abschnitt unten behandelt eine Zeile dieser Tabelle.

## Die Daten

Wir verwenden den klassischen Datensatz `tips`: 244 Restaurantrechnungen mit Rechnungsbetrag,
Trinkgeld und Angaben dazu, wer bezahlt hat (Geschlecht, Raucher, Wochentag, Tageszeit). Er ist in
`data/tips.csv` gespeichert.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import scipy.stats as stats

df = pd.read_csv("data/tips.csv")

# eine CSV speichert Kategorien als reinen Text, daher die natürliche Reihenfolge wiederherstellen:
# sonst sortiert pandas die Tage alphabetisch, Fri, Sat, Sun, Thur
df["day"] = pd.Categorical(df["day"], categories=["Thur", "Fri", "Sat", "Sun"], ordered=True)
df["time"] = pd.Categorical(df["time"], categories=["Lunch", "Dinner"], ordered=True)

df.head()

In [ ]:
# Der Trinkgeldanteil in Prozent ist oft aussagekräftiger als das reine Trinkgeld,
# weil größere Rechnungen automatisch größere Trinkgelder mit sich bringen
df["tip_pct"] = df["tip"] / df["total_bill"] * 100
df.describe()

---
## 1. Numerisch vs. numerisch: Korrelation

**EDA-Frage:** Gehen höhere Rechnungen mit höheren Trinkgeldern einher?

Der Plot für diese Art von Paar ist ein Streudiagramm, die Effektstärke ist der
Korrelationskoeffizient, und der Test fragt, ob dieser Koeffizient plausibel null sein könnte.

In [ ]:
sns.scatterplot(data=df, x="total_bill", y="tip", alpha=0.6)
plt.title("Trinkgeld vs. Rechnungsbetrag")
plt.show()

In [ ]:
# Pearson misst die Stärke eines linearen Zusammenhangs;
# die H0 lautet, dass die wahre Korrelation null ist
r, p_value = stats.pearsonr(df["total_bill"], df["tip"])
print(f"Pearson r: {r:.3f}, p-Wert: {p_value:.2e}")

Das Streudiagramm zeigt einen klaren Aufwärtstrend, und die Zahlen bestätigen ihn: r = 0,68, eine
starke positive Korrelation, mit einem p-Wert von etwa 1e-33. Bei 244 Punkten ist eine so starke
Korrelation unter Unabhängigkeit praktisch unmöglich, dieses Muster ist also echt.

Die Alternative: Die **Spearman-Korrelation** (`stats.spearmanr`) bringt die Daten zuerst in eine
Rangfolge, ist daher robust gegenüber Ausreißern und erfasst jeden monotonen (nicht nur linearen)
Zusammenhang. Hier liefert sie fast dieselbe Antwort (0,68), was uns zeigt, dass der lineare
Zusammenhang nicht von ein paar extremen Rechnungen getragen wird.

Eine Warnung, die kein Test aufhebt: Korrelation ist keine Kausalität. Die Rechnung verursacht das
Trinkgeld nicht von sich aus; die Größe der Gruppe treibt beides an.

---
## 2. Kategorial vs. numerisch, zwei Gruppen: t-Test

**EDA-Frage:** Sind Rechnungen beim Abendessen höher als beim Mittagessen?

Der Plot ist ein Boxplot pro Gruppe, die Effektstärke ist die Differenz der Gruppenmittelwerte,
und der Test ist ein t-Test für zwei Stichproben.

In [ ]:
sns.boxplot(data=df, x="time", y="total_bill")
plt.title("Rechnungsbetrag nach Tageszeit")
plt.show()

df.groupby("time", observed=True)["total_bill"].agg(["count", "mean", "std"])

### Zuerst die Voraussetzungen prüfen

Der t-Test setzt innerhalb jeder Gruppe ungefähr normalverteilte Daten voraus. Rechnungen sind
Geldbeträge, und Geldbeträge sind meist rechtsschief, also prüfen wir das, bevor wir dem Test
vertrauen.

In [ ]:
lunch = df[df["time"] == "Lunch"]["total_bill"]
dinner = df[df["time"] == "Dinner"]["total_bill"]

# Shapiro-Wilk: die H0 lautet, dass die Daten normalverteilt sind
print(f"Shapiro-p-Wert (Mittagessen): {stats.shapiro(lunch)[1]:.2e}")
print(f"Shapiro-p-Wert (Abendessen):  {stats.shapiro(dinner)[1]:.2e}")

# Levene: die H0 lautet, dass beide Gruppen gleiche Varianzen haben
print(f"Levene-p-Wert: {stats.levene(lunch, dinner)[1]:.3f}")

Shapiro-Wilk verwirft die Normalverteilung für beide Gruppen (p-Werte um 1e-06): Die Rechnungen
sind wie erwartet rechtsschief. Das ist hier nicht fatal, denn bei 68 und 176 Beobachtungen sorgt
der zentrale Grenzwertsatz aus Notebook 02 dafür, dass die Stichprobenverteilung des Durchschnitts
jeder Gruppe trotzdem ungefähr normal ist. Es ist aber ein Grund, mit einem Test nachzuprüfen, der
keine Normalverteilung voraussetzt, und das tun wir unten.

Wir verwenden den **Welch-t-Test** (`equal_var=False`), der keine gleichen Varianzen voraussetzt.
Der Levene-Test hat gleiche Varianzen nicht verworfen (p = 0,10), aber Welch kostet fast nichts,
wenn die Varianzen gleich sind, und schützt dich, wenn sie es nicht sind. Deshalb ist er die
vernünftige Standardwahl.

In [ ]:
# Welch-t-Test: die H0 lautet, dass Mittag- und Abendessen denselben mittleren Rechnungsbetrag haben
t_stat, p_value = stats.ttest_ind(lunch, dinner, equal_var=False)
print(f"Welch-t-Statistik: {t_stat:.3f}, p-Wert: {p_value:.5f}")

# Robustheitsprüfung ohne Normalverteilungsannahme:
# Mann-Whitney-U vergleicht die beiden Verteilungen über Ränge
u_stat, p_mw = stats.mannwhitneyu(lunch, dinner)
print(f"Mann-Whitney-U-p-Wert: {p_mw:.5f}")

Rechnungen beim Abendessen liegen im Schnitt bei etwa 20,80 gegenüber 17,17 beim Mittagessen, und
der Welch-t-Test liefert p = 0,002: Der Unterschied ist signifikant. Der Mann-Whitney-U-Test, der
überhaupt keine Normalverteilung voraussetzt, stimmt zu (p = 0,001). Wenn ein parametrischer Test
und seine nichtparametrische Alternative in dieselbe Richtung zeigen, ist die Schlussfolgerung
belastbar: Beim Abendessen wird mehr ausgegeben.

### Ein Muster, das nicht besteht: der Trinkgeldanteil

Die reinen Rechnungsbeträge unterscheiden sich je nach Tageszeit. Unterscheidet sich auch der
Trinkgeld*anteil*? Die Gruppenmittelwerte sehen leicht unterschiedlich aus, und genau das ist die
Art von Muster, die dich in der EDA dazu verleitet, darüber zu berichten.

In [ ]:
lunch_pct = df[df["time"] == "Lunch"]["tip_pct"]
dinner_pct = df[df["time"] == "Dinner"]["tip_pct"]
print(f"Mittleres Trinkgeld in %: Mittagessen {lunch_pct.mean():.2f}, Abendessen {dinner_pct.mean():.2f}")

t_stat, p_value = stats.ttest_ind(lunch_pct, dinner_pct, equal_var=False)
print(f"Welch-t-Statistik: {t_stat:.3f}, p-Wert: {p_value:.3f}")

Gäste beim Mittagessen geben im Schnitt 16,4 % Trinkgeld, beim Abendessen 16,0 %, aber der p-Wert
beträgt 0,51: Ein so kleiner Unterschied ist vollkommen mit Zufall vereinbar. Das ist die Lehre
des ganzen Notebooks in einer Zeile: **Der Plot schlägt vor, der Test entscheidet.** Das
Abendessen bringt höhere Rechnungen und höhere Trinkgelder, aber keine großzügigeren
Trinkgeldgeber.

---
## 3. Kategorial vs. numerisch, mehr als zwei Gruppen: ANOVA

**EDA-Frage:** Hängt die durchschnittliche Rechnung vom Wochentag ab?

Bei vier Gruppen (Donnerstag bis Sonntag) würden t-Tests für jedes Paar sechs Tests und sechs
Gelegenheiten für ein falsch-positives Ergebnis bedeuten. Die **einfaktorielle ANOVA** stellt
stattdessen eine einzige Frage: Sind alle Gruppenmittelwerte gleich? Ihre H0 lautet, dass jeder
Tag denselben mittleren Rechnungsbetrag hat. Ihre Teststatistik **F** vergleicht, wie weit die
Gruppenmittelwerte auseinanderliegen, damit, wie stark die Rechnungen innerhalb jeder Gruppe
streuen, und ihr p-Wert ist eine Fläche unter der F-Kurve aus Notebook 01.

In [ ]:
sns.boxplot(data=df, x="day", y="total_bill")
plt.title("Rechnungsbetrag nach Wochentag")
plt.show()

df.groupby("day", observed=True)["total_bill"].mean().round(2)

In [ ]:
# Einfaktorielle ANOVA über die vier Tage
groups = [g["total_bill"].values for _, g in df.groupby("day", observed=True)]
f_stat, p_value = stats.f_oneway(*groups)
print(f"ANOVA-F-Statistik: {f_stat:.3f}, p-Wert: {p_value:.4f}")

# Dieselbe Frage für den Trinkgeldanteil
groups_pct = [g["tip_pct"].values for _, g in df.groupby("day", observed=True)]
f_stat2, p_value2 = stats.f_oneway(*groups_pct)
print(f"ANOVA auf Trinkgeld in %: F-Statistik: {f_stat2:.3f}, p-Wert: {p_value2:.3f}")

Für den Rechnungsbetrag liefert die ANOVA p = 0,043: knapp unter 0,05, getragen von den
Wochenendtagen (Samstag 20,44 und Sonntag 21,41 gegenüber etwa 17,50 am Donnerstag und Freitag).
Behandle einen solchen grenzwertigen p-Wert als Spur, nicht als Urteil; Abschnitt 5 erklärt,
warum.

Für den Trinkgeldanteil ist die Antwort eindeutig in die andere Richtung: p = 0,47. Der
Trinkgeldanteil ist über die Woche hinweg konstant.

Der nächste Schritt: Eine signifikante ANOVA sagt „mindestens ein Tag unterscheidet sich“, aber
nicht, welcher. Um das zu beantworten, braucht es Post-hoc-Paarvergleiche mit einer Korrektur für
multiple Vergleiche (zum Beispiel Tukeys HSD in der Bibliothek `statsmodels`, die dieses
Repository nicht installiert), und keinen Haufen unkorrigierter t-Tests.

---
## 4. Kategorial vs. kategorial: Chi-Quadrat-Unabhängigkeitstest

**EDA-Frage:** Sind Raucher beim Abendessen überrepräsentiert?

Der Plot ist ein gruppierter Countplot, die Zusammenfassung ist eine Kontingenztafel, und der Test
ist der **Chi-Quadrat-Unabhängigkeitstest**, dessen H0 lautet, dass die beiden Variablen nicht
zusammenhängen. Seine Teststatistik misst, wie weit die beobachteten Häufigkeiten von den
Häufigkeiten entfernt liegen, die man erwarten würde, wenn das so wäre, und sein p-Wert ist eine
Fläche unter der Chi-Quadrat-Kurve aus Notebook 01.

In [ ]:
sns.countplot(data=df, x="time", hue="smoker")
plt.title("Raucher nach Tageszeit")
plt.show()

contingency_table = pd.crosstab(df["smoker"], df["time"])
contingency_table

In [ ]:
chi2_stat, p_value, dof, expected_freq = stats.chi2_contingency(contingency_table)
print(f"Chi-Quadrat-Statistik: {chi2_stat:.3f}, p-Wert: {p_value:.4f}, Freiheitsgrade: {dof}")

# Cramérs V: Effektstärke für eine Kontingenztafel, von 0 (keiner) bis 1 (perfekt)
n = contingency_table.to_numpy().sum()
cramers_v = np.sqrt(chi2_stat / (n * (min(contingency_table.shape) - 1)))
print(f"Cramérs V: {cramers_v:.3f}")

Der p-Wert beträgt 0,48 und Cramérs V 0,05, eine Effektstärke nahe null: Rauchen und Tageszeit
hängen in diesen Daten nicht zusammen. Der Countplot kann nur deshalb schief aussehen, weil es
insgesamt mehr Rechnungen beim Abendessen gibt; der Chi-Quadrat-Test vergleicht die beobachteten
Häufigkeiten mit den Häufigkeiten, die aus den Randsummen zu erwarten sind, und lässt sich daher
von Gruppengrößen nicht täuschen.

Es ist wichtig, beide Zahlen zu berichten. Der p-Wert sagt, ob ein Zusammenhang besteht; **Cramérs
V** sagt, wie stark er wäre. Mit einer ausreichend großen Stichprobe wird selbst ein trivialer
Zusammenhang „signifikant“, deshalb hält eine Effektstärke die Schlussfolgerung ehrlich.

---
## 5. Die EDA-Falle: multiples Testen

Dieses Notebook hat sieben Hypothesentests durchgeführt. Bei alpha = 0,05 hat jeder Test, der auf
reinem Rauschen läuft, eine Wahrscheinlichkeit von 5 % für ein falsch-positives Ergebnis. Die
Wahrscheinlichkeit für mindestens einen Fehlalarm wächst also mit jedem zusätzlichen Vergleich
schnell: Die EDA ist genau die Situation, in der du viele Muster testest, weil die Plots dich dazu
eingeladen haben.

Deshalb verdient das grenzwertige Ergebnis zum Wochentag (p = 0,043) Skepsis: Es war einer von
mehreren Vergleichen und würde nicht einmal eine milde Korrektur für multiples Testen überstehen
(eine Bonferroni-Korrektur über unsere sieben Tests würde p < 0,007 verlangen).

Praktische Regeln für das Testen während der EDA:

- **Zähle deine Tests.** Notiere jeden Test, den du durchführst, auch die, die „nicht funktioniert
  haben“.
- **Korrigieren oder bestätigen.** Wende entweder eine Korrektur für multiple Vergleiche an
  (Bonferroni, Holm) oder behandle explorative Ergebnisse als Hypothesen, die an neuen Daten
  bestätigt werden müssen.
- **Berichte Effektstärken zusammen mit p-Werten.** Ein p-Wert allein verrät nicht, ob der Effekt
  überhaupt von Bedeutung ist.
- **Jage nie nach p < 0,05.** Untergruppe um Untergruppe zu testen, bis etwas signifikant ist, ist
  p-Hacking, und solche Ergebnisse lassen sich nicht replizieren.

---
## Zusammenfassung

In diesem Notebook hast du:

- für jede Art von Variablenpaar, die dir in der EDA begegnet, den passenden Test ausgewählt.
- ein Muster zwischen zwei numerischen Variablen mit einem Pearson-Korrelationstest geprüft
  (Trinkgeld vs. Rechnung: r = 0,68, p etwa 1e-33, echt).
- Unterschiede zwischen zwei Gruppen mit dem Welch-t-Test getestet und mit Mann-Whitney-U
  bestätigt (Rechnungen beim Abendessen höher, p = 0,002; Trinkgeldanteil unverändert, p = 0,51).
- vier Gruppen auf einmal mit einer einfaktoriellen ANOVA verglichen (Rechnungen nach Wochentag:
  grenzwertig, p = 0,043; Trinkgeldanteil: p = 0,47).
- einen Zusammenhang zwischen kategorialen Variablen mit dem Chi-Quadrat-Test plus Cramérs V
  getestet (Raucher vs. Tageszeit: p = 0,48, V = 0,05, kein Zusammenhang).
- die Regeln zum multiplen Testen angewendet, die explorative Ergebnisse ehrlich halten.

| EDA-Frage | Art des Paars | Test | p-Wert | Urteil |
|---|---|---|---|---|
| Höhere Rechnungen, höhere Trinkgelder? | numerisch vs. numerisch | Pearson-Korrelation | 1e-33 | Echt |
| Rechnungen beim Abendessen höher? | kategorial (2) vs. numerisch | Welch-t-Test | 0,002 | Echt |
| Gäste beim Abendessen großzügiger? | kategorial (2) vs. numerisch | Welch-t-Test | 0,51 | Nicht gestützt |
| Rechnungen je nach Wochentag unterschiedlich? | kategorial (4) vs. numerisch | Einfaktorielle ANOVA | 0,043 | Grenzwertige Spur |
| Raucher bevorzugen das Abendessen? | kategorial vs. kategorial | Chi-Quadrat | 0,48 | Nicht gestützt |

Als Nächstes gibt dir `05_hypothesis_testing_for_eda_exercise.ipynb` einen unordentlicheren
Datensatz und dieselben Fragen, aber ohne ausgearbeitete Antworten.

## Quellen & weiterführende Literatur

- [**scipy.stats.pearsonr**](https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.pearsonr.html): Korrelationskoeffizient mit Signifikanztest.
- [**scipy.stats.ttest_ind**](https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.ttest_ind.html): t-Test für zwei unabhängige Stichproben (`equal_var=False` für Welch).
- [**scipy.stats.mannwhitneyu**](https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.mannwhitneyu.html): Die rangbasierte Alternative zum t-Test.
- [**scipy.stats.f_oneway**](https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.f_oneway.html): Einfaktorielle ANOVA.
- [**scipy.stats.chi2_contingency**](https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.chi2_contingency.html): Chi-Quadrat-Unabhängigkeitstest.
- [**Statistical Hypothesis Analysis in Python with ANOVAs, Chi-Square, and Pearson Correlation**](https://stackabuse.com/statistical-hypothesis-analysis-in-python-with-anovas-chi-square-and-pearson-correlation/): Dieselben drei Testfamilien an einem anderen Datensatz durchgerechnet.
- [**Codecademy: Testing an Association cheat sheet**](https://www.codecademy.com/learn/hypothesis-testing-python/modules/hp-testing-an-association/cheatsheet): Einseitige Übersicht, welcher Test zu welchem Paar passt.
- [**StatQuest: P-hacking**](https://www.youtube.com/watch?v=HDCOUXE3HMM): Warum es die Regeln zum multiplen Testen aus Abschnitt 5 gibt.